<a href="https://colab.research.google.com/github/Renyambar/Reny_MachineLearning/blob/main/JS09/JS09_Tugas2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#TUGAS LAB 2 — Spotify Nearest Track Search

##Install & Import Library

In [1]:
!pip install -q kagglehub annoy faiss-cpu hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [29]:
import os
import glob
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import kagglehub
from annoy import AnnoyIndex
import faiss
import hnswlib

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")

print("Semua library berhasil di-import.")

Semua library berhasil di-import.


##Download Dataset

In [30]:
dataset_path = kagglehub.dataset_download(
    "bwandowando/spotify-songs-with-attributes-and-lyrics"
)

print("Dataset berhasil diunduh.")
print("Lokasi dataset:", dataset_path)

Using Colab cache for faster access to the 'spotify-songs-with-attributes-and-lyrics' dataset.
Dataset berhasil diunduh.
Lokasi dataset: /kaggle/input/spotify-songs-with-attributes-and-lyrics


In [31]:
csv_files = glob.glob(
    os.path.join(dataset_path, "**", "*.csv"),
    recursive=True
)

print("Daftar file CSV:")

for i, file in enumerate(csv_files, start=1):
    print(f"{i}. {os.path.basename(file)}")

Daftar file CSV:
1. songs_with_lyrics_and_timestamps.csv
2. songs_with_attributes_and_lyrics.csv


In [32]:
DATA_PATH = None

for file in csv_files:
    if os.path.basename(file) == "songs_with_attributes_and_lyrics.csv":
        DATA_PATH = file
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "File songs_with_attributes_and_lyrics.csv tidak ditemukan."
    )

print("Dataset yang digunakan:")
print(DATA_PATH)

Dataset yang digunakan:
/kaggle/input/spotify-songs-with-attributes-and-lyrics/songs_with_attributes_and_lyrics.csv


## Membaca Datasset

In [33]:
df = pd.read_csv(DATA_PATH)

print("Dataset berhasil dimuat.")
print("Jumlah baris dan kolom:", df.shape)

display(df.head())

Dataset berhasil dimuat.
Jumlah baris dan kolom: (955320, 17)


,id,name,album_name,artists,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,lyrics
0,0Prct5TDjAnEgIqbxcldY9,!,UNDEN!ABLE,HELLYEAH,0.415,0.6050,7,-11.157,1,0.0575,0.00116,0.838000,0.4710,0.193,100.059,79500.0,"He said he came from Jamaica,\n he owned a cou..."
1,2ASl4wirkeYm3OWZxXKYuq,!!,NaN,Yxngxr1,0.788,0.6480,7,-9.135,0,0.3150,0.90000,0.000000,0.1760,0.287,79.998,114000.0,"Fucked a bitch, now she running with my kids\n..."
2,69lcggVPmOr9cvPx9kLiiN,!!! - Interlude,Where I Belong EP,Glowie,0.000,0.0354,7,-20.151,0,0.0000,0.90800,0.000000,0.4790,0.000,0.000,11413.0,"Oh, my God, I'm going crazy\n"
3,4U7dlZjg1s9pjdppqZy0fm,!!De Repente!!,Un Palo Al Agua (20 Grandes Canciones),Rosendo,0.657,0.8820,5,-6.340,1,0.0385,0.00740,0.000013,0.0474,0.939,123.588,198173.0,Continuamente se extraña la gente si no puede ...
4,4v1IBp3Y3rpkWmWzIlkYju,!!De Repente!!,Fuera De Lugar,Rosendo,0.659,0.8930,5,-8.531,1,0.0411,0.09220,0.000019,0.0534,0.951,123.600,199827.0,Continuamente se extraña la gente si no puede ...


In [34]:
print("Informasi dataset:")
df.info()

print("\nDaftar kolom:")
print(df.columns.tolist())

print("\nJumlah data kosong:")
display(df.isnull().sum())

Informasi dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 955320 entries, 0 to 955319
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                955320 non-null  object 
 1   name              955309 non-null  object 
 2   album_name        385557 non-null  object 
 3   artists           955315 non-null  object 
 4   danceability      955320 non-null  float64
 5   energy            955320 non-null  float64
 6   key               955320 non-null  object 
 7   loudness          955320 non-null  float64
 8   mode              955320 non-null  object 
 9   speechiness       955320 non-null  float64
 10  acousticness      955320 non-null  float64
 11  instrumentalness  955320 non-null  float64
 12  liveness          955320 non-null  float64
 13  valence           955320 non-null  float64
 14  tempo             955320 non-null  float64
 15  duration_ms       955320 non-null  float64
 16  l

,0
id,0
name,11
album_name,569763
artists,5
danceability,0
energy,0
key,0
loudness,0
mode,0
speechiness,0


## Memilih Fitur

In [35]:
features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo"
]

missing_features = [
    col for col in features if col not in df.columns
]

if missing_features:
    raise ValueError(
        f"Fitur berikut tidak ditemukan: {missing_features}"
    )

X_df = df[features].copy()

print("Fitur yang digunakan:")
print(features)

display(X_df.head())

Fitur yang digunakan:
['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']


,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,0.415,0.6050,-11.157,0.0575,0.00116,0.838000,0.4710,0.193,100.059
1,0.788,0.6480,-9.135,0.3150,0.90000,0.000000,0.1760,0.287,79.998
2,0.000,0.0354,-20.151,0.0000,0.90800,0.000000,0.4790,0.000,0.000
3,0.657,0.8820,-6.340,0.0385,0.00740,0.000013,0.0474,0.939,123.588
4,0.659,0.8930,-8.531,0.0411,0.09220,0.000019,0.0534,0.951,123.600


##Membersihkan Data

In [36]:
X_df = X_df.apply(pd.to_numeric, errors="coerce")

valid_mask = (
    X_df.notna().all(axis=1) &
    np.isfinite(X_df).all(axis=1)
)

df_clean = df.loc[valid_mask].copy().reset_index(drop=True)
X_df = X_df.loc[valid_mask].reset_index(drop=True)

print("Jumlah data sebelum preprocessing:", len(df))
print("Jumlah data setelah preprocessing:", len(df_clean))
print("Jumlah data yang dihapus:", len(df) - len(df_clean))

Jumlah data sebelum preprocessing: 955320
Jumlah data setelah preprocessing: 955320
Jumlah data yang dihapus: 0


##Normalisasi

In [37]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_df)

X_scaled = np.ascontiguousarray(
    X_scaled,
    dtype=np.float32
)

print("Normalisasi selesai.")
print("Shape data:", X_scaled.shape)
print("Tipe data:", X_scaled.dtype)

Normalisasi selesai.
Shape data: (955320, 9)
Tipe data: float32


In [38]:
K = 10
QUERY_INDEX = 0

N = len(X_scaled)
DIM = X_scaled.shape[1]

if N <= K:
    raise ValueError("Jumlah data harus lebih besar dari K.")

if QUERY_INDEX < 0 or QUERY_INDEX >= N:
    raise ValueError("QUERY_INDEX berada di luar jumlah data.")

query_vector = X_scaled[QUERY_INDEX].reshape(1, -1)

query_song = df_clean.iloc[QUERY_INDEX]

print("Lagu yang dijadikan query:")
print("Nama lagu :", query_song["name"])
print("Artis     :", query_song["artists"])
print("Album     :", query_song["album_name"])
print("\nJumlah data:", N)
print("Dimensi fitur:", DIM)
print("Jumlah rekomendasi:", K)

Lagu yang dijadikan query:
Nama lagu : !
Artis     : HELLYEAH
Album     : UNDEN!ABLE

Jumlah data: 955320
Dimensi fitur: 9
Jumlah rekomendasi: 10


In [39]:
def get_song_information(indices, distances, algorithm):
    results = []

    for rank, (idx, distance) in enumerate(
        zip(indices, distances),
        start=1
    ):
        song = df_clean.iloc[int(idx)]

        results.append({
            "Ranking": rank,
            "Nama Lagu": song["name"],
            "Artis": song["artists"],
            "Album": song["album_name"],
            "Distance": float(distance),
            "Index": int(idx),
            "Algoritma": algorithm
        })

    return pd.DataFrame(results)

##Exact NN

In [40]:
exact_model = NearestNeighbors(
    n_neighbors=K + 1,
    algorithm="brute",
    metric="euclidean",
    n_jobs=-1
)

# Mengukur waktu pembangunan model
start_time = time.perf_counter()

exact_model.fit(X_scaled)

exact_build_time = time.perf_counter() - start_time

# Mengukur waktu pencarian
start_time = time.perf_counter()

exact_distances, exact_indices = exact_model.kneighbors(
    query_vector
)

exact_query_time = time.perf_counter() - start_time

# Menghilangkan lagu query dari hasil
exact_results = [
    (int(idx), float(dist))
    for idx, dist in zip(
        exact_indices[0],
        exact_distances[0]
    )
    if idx != QUERY_INDEX
][:K]

exact_ids = [idx for idx, dist in exact_results]
exact_dists = [dist for idx, dist in exact_results]

exact_table = get_song_information(
    exact_ids,
    exact_dists,
    "Exact NN"
)

print("Exact Nearest Neighbors selesai.")
print("Build time:", round(exact_build_time, 6), "detik")
print("Query time:", round(exact_query_time, 6), "detik")

display(exact_table)

Exact Nearest Neighbors selesai.
Build time: 0.006392 detik
Query time: 0.041489 detik


,Ranking,Nama Lagu,Artis,Album,Distance,Index,Algoritma
0,1,Intro (Who The Fuck Are You'),C-Lekktor,"Silence Procession, The",0.886657,394553,Exact NN
1,2,Testimonium,Lacrimosa,Testimonium,0.909443,764272,Exact NN
2,3,Ultimate Satisfaction - The Field Remix,Bear In Heaven,NaN,0.936008,837727,Exact NN
3,4,Sæglópur,Sigur Rós,NaN,0.937600,749223,Exact NN
4,5,Autumn Interlude,Amethystium,NaN,0.946505,61511,Exact NN
5,6,Púrpura,Margaritas Podridas,NaN,0.976332,625227,Exact NN
6,7,Turn from Heaven,Therion,NaN,0.995418,833164,Exact NN
7,8,See You When I Die,BOYO,NaN,1.020761,676733,Exact NN
8,9,Mrityu,Multani;Karun;Udbhav,Old World Songs in a New World Order,1.040783,523698,Exact NN
9,10,Bewitched,The Wedding Present,Bizarro,1.045661,85956,Exact NN


##ANNOY

In [41]:
annoy_index = AnnoyIndex(DIM, "euclidean")

start_time = time.perf_counter()

for i in range(N):
    annoy_index.add_item(i, X_scaled[i].tolist())

annoy_index.build(10)

annoy_build_time = time.perf_counter() - start_time

print("Index ANNOY berhasil dibuat.")
print("Build time:", round(annoy_build_time, 6), "detik")

Index ANNOY berhasil dibuat.
Build time: 15.983379 detik


In [42]:
start_time = time.perf_counter()

annoy_indices, annoy_distances = annoy_index.get_nns_by_vector(
    query_vector[0].tolist(),
    K + 1,
    search_k=-1,
    include_distances=True
)

annoy_query_time = time.perf_counter() - start_time

# Menghilangkan lagu query
annoy_results = [
    (int(idx), float(dist))
    for idx, dist in zip(
        annoy_indices,
        annoy_distances
    )
    if idx != QUERY_INDEX
][:K]

annoy_ids = [idx for idx, dist in annoy_results]
annoy_dists = [dist for idx, dist in annoy_results]

annoy_table = get_song_information(
    annoy_ids,
    annoy_dists,
    "ANNOY"
)

print("Pencarian ANNOY selesai.")
print("Query time:", round(annoy_query_time, 6), "detik")

display(annoy_table)

Pencarian ANNOY selesai.
Query time: 0.000432 detik


,Ranking,Nama Lagu,Artis,Album,Distance,Index,Algoritma
0,1,Testimonium,Lacrimosa,Testimonium,0.909443,764272,ANNOY
1,2,Sæglópur,Sigur Rós,NaN,0.937600,749223,ANNOY
2,3,Púrpura,Margaritas Podridas,NaN,0.976332,625227,ANNOY
3,4,Turn from Heaven,Therion,NaN,0.995418,833164,ANNOY
4,5,See You When I Die,BOYO,NaN,1.020761,676733,ANNOY
5,6,Mrityu,Multani;Karun;Udbhav,Old World Songs in a New World Order,1.040783,523698,ANNOY
6,7,Bewitched,The Wedding Present,Bizarro,1.045661,85956,ANNOY
7,8,Who Makes the Nazis? - Live - Bonus Track,The Fall,Fall in a Hole (Expanded Edition),1.048527,888081,ANNOY
8,9,Teri Yaadon Se - Lofi Flip,Jammy Weirdo;Mustafa Zahid;Pranay.M. Rijia,Teri Yaadon Se (Lofi Flip),1.051962,763625,ANNOY
9,10,Yngve-Frej,Torulf,NaN,1.056078,906004,ANNOY


##FAISS

In [43]:
nlist = min(100, max(1, N // 40))

quantizer = faiss.IndexFlatL2(DIM)

faiss_index = faiss.IndexIVFFlat(
    quantizer,
    DIM,
    nlist,
    faiss.METRIC_L2
)

start_time = time.perf_counter()

faiss_index.train(X_scaled)
faiss_index.add(X_scaled)

faiss_build_time = time.perf_counter() - start_time

faiss_index.nprobe = min(10, nlist)

print("Index FAISS berhasil dibuat.")
print("Jumlah cluster:", nlist)
print("Jumlah cluster yang diperiksa:", faiss_index.nprobe)
print("Build time:", round(faiss_build_time, 6), "detik")

Index FAISS berhasil dibuat.
Jumlah cluster: 100
Jumlah cluster yang diperiksa: 10
Build time: 0.283317 detik


In [44]:
start_time = time.perf_counter()

faiss_distances, faiss_indices = faiss_index.search(
    query_vector,
    K + 1
)

faiss_query_time = time.perf_counter() - start_time

# Menghilangkan lagu query
faiss_results = [
    (int(idx), float(np.sqrt(max(dist, 0))))
    for idx, dist in zip(
        faiss_indices[0],
        faiss_distances[0]
    )
    if idx >= 0 and idx != QUERY_INDEX
][:K]

faiss_ids = [idx for idx, dist in faiss_results]
faiss_dists = [dist for idx, dist in faiss_results]

faiss_table = get_song_information(
    faiss_ids,
    faiss_dists,
    "FAISS IVF-Flat"
)

print("Pencarian FAISS selesai.")
print("Query time:", round(faiss_query_time, 6), "detik")

display(faiss_table)

Pencarian FAISS selesai.
Query time: 0.011404 detik


,Ranking,Nama Lagu,Artis,Album,Distance,Index,Algoritma
0,1,Intro (Who The Fuck Are You'),C-Lekktor,"Silence Procession, The",0.886657,394553,FAISS IVF-Flat
1,2,Testimonium,Lacrimosa,Testimonium,0.909443,764272,FAISS IVF-Flat
2,3,Ultimate Satisfaction - The Field Remix,Bear In Heaven,NaN,0.936008,837727,FAISS IVF-Flat
3,4,Sæglópur,Sigur Rós,NaN,0.937600,749223,FAISS IVF-Flat
4,5,Autumn Interlude,Amethystium,NaN,0.946505,61511,FAISS IVF-Flat
5,6,Púrpura,Margaritas Podridas,NaN,0.976332,625227,FAISS IVF-Flat
6,7,Turn from Heaven,Therion,NaN,0.995418,833164,FAISS IVF-Flat
7,8,See You When I Die,BOYO,NaN,1.020761,676733,FAISS IVF-Flat
8,9,Mrityu,Multani;Karun;Udbhav,Old World Songs in a New World Order,1.040783,523698,FAISS IVF-Flat
9,10,Bewitched,The Wedding Present,Bizarro,1.045661,85956,FAISS IVF-Flat


##HNSW

In [45]:
hnsw_index = hnswlib.Index(
    space="l2",
    dim=DIM
)

start_time = time.perf_counter()

hnsw_index.init_index(
    max_elements=N,
    ef_construction=200,
    M=16
)

hnsw_index.add_items(
    X_scaled,
    np.arange(N)
)

hnsw_index.set_ef(50)

hnsw_build_time = time.perf_counter() - start_time

print("Index HNSW berhasil dibuat.")
print("Build time:", round(hnsw_build_time, 6), "detik")

Index HNSW berhasil dibuat.
Build time: 189.830489 detik


In [46]:
start_time = time.perf_counter()

hnsw_indices, hnsw_distances = hnsw_index.knn_query(
    query_vector,
    k=K + 1
)

hnsw_query_time = time.perf_counter() - start_time

# Menghilangkan lagu query
hnsw_results = [
    (int(idx), float(np.sqrt(max(dist, 0))))
    for idx, dist in zip(
        hnsw_indices[0],
        hnsw_distances[0]
    )
    if idx != QUERY_INDEX
][:K]

hnsw_ids = [idx for idx, dist in hnsw_results]
hnsw_dists = [dist for idx, dist in hnsw_results]

hnsw_table = get_song_information(
    hnsw_ids,
    hnsw_dists,
    "HNSW"
)

print("Pencarian HNSW selesai.")
print("Query time:", round(hnsw_query_time, 6), "detik")

display(hnsw_table)

Pencarian HNSW selesai.
Query time: 0.001503 detik


,Ranking,Nama Lagu,Artis,Album,Distance,Index,Algoritma
0,1,Intro (Who The Fuck Are You'),C-Lekktor,"Silence Procession, The",0.886657,394553,HNSW
1,2,Testimonium,Lacrimosa,Testimonium,0.909443,764272,HNSW
2,3,Ultimate Satisfaction - The Field Remix,Bear In Heaven,NaN,0.936008,837727,HNSW
3,4,Sæglópur,Sigur Rós,NaN,0.937600,749223,HNSW
4,5,Autumn Interlude,Amethystium,NaN,0.946505,61511,HNSW
5,6,Púrpura,Margaritas Podridas,NaN,0.976332,625227,HNSW
6,7,Turn from Heaven,Therion,NaN,0.995418,833164,HNSW
7,8,See You When I Die,BOYO,NaN,1.020761,676733,HNSW
8,9,Mrityu,Multani;Karun;Udbhav,Old World Songs in a New World Order,1.040783,523698,HNSW
9,10,Bewitched,The Wedding Present,Bizarro,1.045661,85956,HNSW


##PERBANDINGAN

In [50]:
def calculate_recall(exact_ids, approximate_ids):
    exact_set = set(exact_ids)
    approximate_set = set(approximate_ids)

    if len(exact_set) == 0:
        return 0.0

    correct = len(exact_set.intersection(approximate_set))

    return correct / len(exact_set)

annoy_recall = calculate_recall(
    exact_ids,
    annoy_ids
)

faiss_recall = calculate_recall(
    exact_ids,
    faiss_ids
)

hnsw_recall = calculate_recall(
    exact_ids,
    hnsw_ids
)

print("Hasil Recall@K")
print("---------------------------")
print("ANNOY :", round(annoy_recall, 4))
print("FAISS :", round(faiss_recall, 4))
print("HNSW  :", round(hnsw_recall, 4))

Hasil Recall@K
---------------------------
ANNOY : 0.7
FAISS : 1.0
HNSW  : 1.0


In [51]:
comparison_df = pd.DataFrame({
    "Algorithm": [
        "Exact NN",
        "ANNOY",
        "FAISS IVF-Flat",
        "HNSW"
    ],
    "Build Time (s)": [
        exact_build_time,
        annoy_build_time,
        faiss_build_time,
        hnsw_build_time
    ],
    "Query Time (s)": [
        exact_query_time,
        annoy_query_time,
        faiss_query_time,
        hnsw_query_time
    ],
    "Recall@K": [
        1.0,
        annoy_recall,
        faiss_recall,
        hnsw_recall
    ]
})

display(
    comparison_df.style.format({
        "Build Time (s)": "{:.6f}",
        "Query Time (s)": "{:.6f}",
        "Recall@K": "{:.4f}"
    })
)

,Algorithm,Build Time (s),Query Time (s),Recall@K
0,Exact NN,0.006392,0.041489,1.0000
1,ANNOY,15.983379,0.000432,0.7000
2,FAISS IVF-Flat,0.283317,0.011404,1.0000
3,HNSW,189.830489,0.001503,1.0000
